# 07 — Prima Random Forest con pesi

Una Random Forest combina molti alberi. Qui proviamo una sola configurazione con100 alberi, profondità15 e pesi balanced; n_jobs=2 limita il parallelismo. Non è il protocollo originale del paper.
Cinque split identici al notebook 04 (seed 1–5), tutte le 41 feature,
profondità15. Nessun nuovo iperparametro cercato dopo le misure.
Leggiamo soltanto KDDTrain+ e i risultati di riferimento locali.


In [ ]:
from pathlib import Path
import sys
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, FAMILY_ORDER


## 1. Leggere X/y e verificare la sorgente

pd.read_csv e Series.map restano visibili; schema e mapping sono costanti locali.
L'hash e i seed collegano questa prova al confronto precedente.


In [ ]:
train_path = project_root / 'data/raw/KDDTrain+.csv'
string_types = {column: 'string' for column in (*CATEGORICAL_COLUMNS, 'attack_label')}
data = pd.read_csv(train_path, header=None, names=COLUMN_NAMES, dtype=string_types)
unknown = data['attack_label'].isna() | ~data['attack_label'].isin(ATTACK_TO_FAMILY)
if unknown.any():
    raise ValueError('Etichette mancanti o non mappate: riesaminare il target.')
y = data['attack_label'].map(ATTACK_TO_FAMILY).rename('family')
X = data.loc[:, list(FEATURE_COLUMNS)].copy()
assert X.index.equals(pd.RangeIndex(len(X))) and X.shape[1] == 41
reference_path = project_root / 'artifacts/rare_class_reference'
reference = json.loads((reference_path / 'results.json').read_text(encoding='utf-8'))
assert hashlib.sha256(train_path.read_bytes()).hexdigest() == reference['source_sha256']
assert reference['seeds'] == list(range(1, 6)) and reference['feature_count'] == 41


## 2. Definire preprocessing e classificatore

One-hot per le tre categoriche, 38 numeriche invariate. La Pipeline viene
clonata prima di ogni fit. class_weight="balanced" calcola i pesi dal solo y_train:
n_train / (numero_classi × frequenza_classe_train). Non crea campioni sintetici.
compute_class_weight serve qui a rendere visibili i pesi che il classificatore
calcola durante il fit; non li passiamo una seconda volta come sample_weight.


In [ ]:
from time import perf_counter
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier

numeric_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), list(CATEGORICAL_COLUMNS)),
    ('numeric', 'passthrough', numeric_columns),
], remainder='drop')
model_template = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', max_features='sqrt', bootstrap=True, n_jobs=2)),
])
seeds = list(range(1, 6))
depth = 15
model_id = 'random_forest_balanced'


## 3. Cinque split abbinati: fit e predict

Ogni split viene confrontato con le posizioni e le risposte del notebook 04.
Solo dopo creiamo il modello, lo addestriamo sul train e prevediamo la validation.
Controlliamo anche il vocabolario appreso dall'encoder e registriamo pesi,
parametri effettivi e tempi. I tempi descrivono queste esecuzioni sulla macchina.


In [ ]:
prediction_arrays = {}
split_records = []
weight_rows = []
category_rows = []
parameters_by_seed = {}
fit_seconds_by_seed = {}

for run, seed in enumerate(seeds, start=1):
    X_train, X_validation, y_train, y_validation = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed,
    )
    train_indices = X_train.index.to_numpy()
    validation_indices = X_validation.index.to_numpy()
    assert not np.intersect1d(train_indices, validation_indices).size
    with np.load(reference_path / 'predictions.npz', allow_pickle=False) as saved:
        np.testing.assert_array_equal(validation_indices, saved[f'run_{run:02d}_validation_indices'])
        np.testing.assert_array_equal(y_validation.to_numpy(), saved[f'run_{run:02d}_truth'])

    weights = compute_class_weight(class_weight='balanced', classes=np.asarray(FAMILY_ORDER), y=y_train)
    for family, weight in zip(FAMILY_ORDER, weights):
        weight_rows.append({'run': run, 'seed': seed, 'family': family,
                            'train_count': int(y_train.eq(family).sum()), 'weight': float(weight)})
    model = clone(model_template).set_params(classifier__random_state=seed)
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds_by_seed[str(seed)] = perf_counter() - start
    y_pred = model.predict(X_validation)
    parameters_by_seed[str(seed)] = model.named_steps['classifier'].get_params()

    encoder = model.named_steps['preprocessor'].named_transformers_['categorical']
    for column, learned in zip(CATEGORICAL_COLUMNS, encoder.categories_):
        assert set(learned) == set(X_train[column].unique())
        category_rows.append({'run': run, 'feature': column, 'learned_categories': len(learned),
                              'unseen_validation': sorted(set(X_validation[column]) - set(learned))})
    prediction_arrays[f'run_{run:02d}_validation_indices'] = validation_indices
    prediction_arrays[f'run_{run:02d}_truth'] = np.asarray(y_validation, dtype=str)
    prediction_arrays[f'run_{run:02d}_depth_15'] = np.asarray(y_pred, dtype=str)
    split_records.append({
        'run': run, 'seed': seed, 'train_rows': len(X_train), 'validation_rows': len(X_validation),
        'train_indices_sha256': hashlib.sha256(train_indices.tobytes()).hexdigest(),
        'validation_indices_sha256': hashlib.sha256(validation_indices.tobytes()).hexdigest(),
    })
    print(f'{model_id}: run {run}/{len(seeds)} completata (seed={seed}).')

assert split_records == reference['splits']
weight_table = pd.DataFrame(weight_rows)
display(weight_table.loc[weight_table['run'].eq(1)])


## 4. Valutare le predizioni con le funzioni originali sklearn

Prima misuriamo ciascuna run; poi calcoliamo media, deviazione standard campionaria,
minimo e massimo. Riportiamo precision insieme a recall: recuperare più attacchi
può aumentare i falsi positivi. Le cinque validation si sovrappongono.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

run_rows = []
class_rows = []
count_matrices = []
normalized_matrices = []
for run, seed in enumerate(seeds, start=1):
    truth = prediction_arrays[f'run_{run:02d}_truth']
    predicted = prediction_arrays[f'run_{run:02d}_depth_15']
    report = classification_report(truth, predicted, labels=FAMILY_ORDER, output_dict=True, zero_division=0)
    counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
    normalized_matrices.append(confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true'))
    count_matrices.append(counts)
    run_rows.append({
        'run': run, 'seed': seed, 'max_depth': depth,
        'accuracy': accuracy_score(truth, predicted),
        'macro_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0),
    })
    for i, family in enumerate(FAMILY_ORDER):
        support, correct = int(counts[i].sum()), int(counts[i, i])
        class_rows.append({
            'run': run, 'seed': seed, 'max_depth': depth, 'famiglia': family,
            'support': support, 'corretti': correct, 'errori': support - correct,
            'falsi_positivi': int(counts[:, i].sum() - correct),
            'quota_errori': (support - correct) / support,
            'precision': report[family]['precision'], 'recall': report[family]['recall'], 'f1': report[family]['f1-score'],
        })
run_results = pd.DataFrame(run_rows)
class_results = pd.DataFrame(class_rows)
overall_summary = run_results[['accuracy', 'macro_f1', 'weighted_f1']].agg(['mean', 'std', 'min', 'max'])
class_summary = class_results.groupby('famiglia', sort=False)[
    ['quota_errori', 'precision', 'recall', 'f1', 'falsi_positivi']
].agg(['mean', 'std', 'min', 'max'])
display(overall_summary.round(6))
display(class_summary.loc[['R2L', 'U2R']].round(4))


## 5. Salvare il confronto senza modificare il riferimento

Manteniamo lo schema degli artefatti del notebook 04, aggiungendo identità del
modello, pesi e parametri per seed. Nessuna serializzazione dei modelli e nessun
risultato scartato perché sfavorevole. NPZ Unicode rileggibile senza pickle.


In [ ]:
import platform
import importlib.metadata

artifacts = project_root / 'artifacts' / model_id
artifacts.mkdir(parents=True, exist_ok=True)
run_results.to_csv(artifacts / 'run_metrics.csv', index=False)
class_results.to_csv(artifacts / 'class_metrics.csv', index=False)
weight_table.to_csv(artifacts / 'class_weights.csv', index=False)
pd.DataFrame(category_rows).to_json(artifacts / 'encoder_checks.json', orient='records', indent=2)
overall_summary.to_csv(artifacts / 'overall_summary.csv')
class_summary.to_csv(artifacts / 'class_summary.csv')
np.savez_compressed(artifacts / 'predictions.npz', **prediction_arrays)
mean_matrix = np.stack(normalized_matrices).mean(axis=0)
std_matrix = np.stack(normalized_matrices).std(axis=0, ddof=1)
experiment = {
    'model_id': model_id, 'evaluation_set': 'repeated validation from KDDTrain+ only',
    'source': reference['source'], 'source_sha256': reference['source_sha256'],
    'rows': len(X), 'feature_count': X.shape[1], 'validation_size': 0.2,
    'seeds': seeds, 'depths': [depth], 'stratified': True,
    'family_order': list(FAMILY_ORDER), 'std_ddof': 1,
    'randomness_varied': ['train_validation_split', 'classifier'],
    'versions': {'python': platform.python_version(), **{
        package: importlib.metadata.version(package) for package in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']
    }},
    'splits': split_records, 'classifier_parameters_by_seed': parameters_by_seed,
    'fit_seconds_by_seed': fit_seconds_by_seed,
    'run_metrics': run_results.to_dict(orient='records'),
    'class_metrics': class_results.to_dict(orient='records'),
    'normalized_confusion_mean': {'15': mean_matrix.tolist()},
    'normalized_confusion_std': {'15': std_matrix.tolist()},
    'confusion_counts_by_depth': {'15': [matrix.tolist() for matrix in count_matrices]},
    'class_weights': weight_table.to_dict(orient='records'),
    'unique_validation_counts': reference['unique_validation_counts'],
    'limits': [message for message in reference['limits'] if not message.startswith('Exploration of three depths')]
              + ['One class-weighted configuration at max_depth=15; paired repeated holdouts.'],
}
(artifacts / 'results.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')
print('Salvate cinque run; il riferimento non è stato riaddestrato.')


## Lettura degli esiti

Valutare precision, recall, falsi positivi e variabilità insieme. Un miglioramento
di recall accompagnato da un forte calo di precision è un compromesso da discutere,
non una promozione automatica del modello. Il notebook 08 affianca le tre configurazioni.
Nessuna stima del test ufficiale o promessa di generalizzazione a reti nuove.


## Esito verificato

Sui cinque split la Random Forest raggiunge precision U2R media 78,34% e recall 60%,
contro49,62%/40% del riferimento non pesato. Il macro F1 è0,920033±0,031215.
L'errore U2R medio resta40%, con range20–70% fra run: un risultato promettente
nel confronto di sviluppo, ancora fragile sui pochi esempi disponibili.
Questa configurazione resta un candidato da discutere, senza selezione automatica.
